In [1]:
import pandas as pd

print("📥 Загружаем FINAL_READY...")
df = pd.read_csv('rank_dataset_FINAL_READY.csv')

print("\n📊 Баланс классов во всем гигантском файле:")
print(df['target'].value_counts())

# Проверим, нет ли пустых значений
print("\n🔍 Проверка на пропуски (NaN):")
print(df.isnull().sum())

📥 Загружаем FINAL_READY...

📊 Баланс классов во всем гигантском файле:
target
0    9920716
1       2844
Name: count, dtype: int64

🔍 Проверка на пропуски (NaN):
bert_score       0
candidate_id     0
is_same_group    0
target           0
eq_clean         0
level_idx        0
goal_clean       0
dtype: int64


In [2]:
print("Начинаем сборку чистого датасета...")

# 1. Забираем ВСЕ реальные примеры (target = 1)
df_positives = df[df['target'] == 1].copy()

# 2. Берем случайные 50 000 нулей (этого хватит за глаза, чтобы показать границы)
df_negatives = df[df['target'] == 0].sample(n=50000, random_state=42).copy()

# 3. Склеиваем и хорошенько перемешиваем (frac=1)
df_train = pd.concat([df_positives, df_negatives]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅ Готово! Итоговый размер датасета: {len(df_train)} строк.")
print(f"⚖️ Новый баланс (Нули / Единицы):\n{df_train['target'].value_counts()}")

Начинаем сборку чистого датасета...
✅ Готово! Итоговый размер датасета: 52844 строк.
⚖️ Новый баланс (Нули / Единицы):
target
0    50000
1     2844
Name: count, dtype: int64


In [3]:
# Указываем, какие колонки содержат текст/категории
cat_features = ['candidate_id', 'eq_clean', 'goal_clean']

# Заполняем возможные пропуски словом 'Unknown' (Неизвестно), чтобы CatBoost не ругался
for col in cat_features:
    df_train[col] = df_train[col].fillna('Unknown')
    # Принудительно делаем их строками (string)
    df_train[col] = df_train[col].astype(str)

print("🛠 Текстовые (категориальные) признаки обработаны и готовы к бою!")
display(df_train.head())

🛠 Текстовые (категориальные) признаки обработаны и готовы к бою!


,bert_score,candidate_id,is_same_group,target,eq_clean,level_idx,goal_clean
0,0.001278,1546,0,0,Machine,3,Powerbuilding
1,0.001276,1440,0,0,Machine,0,Bodybuilding
2,0.001778,2733,0,0,Machine,0,Athletics
3,0.001300,2731,0,0,Dumbbell,0,Athletics
4,0.001246,1075,0,0,Barbell,1,Athletics


In [4]:
from sklearn.model_selection import train_test_split

# Отделяем признаки (X) от ответов (y)
X = df_train.drop('target', axis=1)
y = df_train['target']

# Разбиваем: 80% на обучение, 20% на экзамен
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y # Сохраняем пропорции 0 и 1 в обеих частях
)

print(f"🏋️ Тренировочная выборка: {len(X_train)} строк")
print(f"🧪 Тестовая (экзамен): {len(X_test)} строк")

🏋️ Тренировочная выборка: 42275 строк
🧪 Тестовая (экзамен): 10569 строк


In [5]:
import pandas as pd
from IPython.display import display

print("🔍 АНАЛИЗ РАСПРЕДЕЛЕНИЙ В ТРЕНИРОВОЧНОЙ ВЫБОРКЕ (df_train)")

# 1. Анализ категориальных колонок
categorical_cols = ['eq_clean', 'level_idx', 'goal_clean', 'is_same_group']

for col in categorical_cols:
    print(f"\n{'='*50}")
    print(f"📊 Распределение признака: {col.upper()}")
    
    # Считаем проценты встречаемости категорий для Единиц и Нулей
    pos_dist = df_train[df_train['target'] == 1][col].value_counts(normalize=True) * 100
    neg_dist = df_train[df_train['target'] == 0][col].value_counts(normalize=True) * 100
    
    # Склеиваем в одну красивую таблицу
    dist_df = pd.DataFrame({
        'Target = 1 (Реальные) %': pos_dist,
        'Target = 0 (Сгенерированные) %': neg_dist
    }).fillna(0).round(2) # Заполняем нулями то, чего нет, и округляем
    
    display(dist_df)

# 2. Анализ непрерывной колонки bert_score
print(f"\n{'='*50}")
print("📈 Статистика для BERT_SCORE:")
display(df_train.groupby('target')['bert_score'].describe().round(5))

🔍 АНАЛИЗ РАСПРЕДЕЛЕНИЙ В ТРЕНИРОВОЧНОЙ ВЫБОРКЕ (df_train)

📊 Распределение признака: EQ_CLEAN


,Target = 1 (Реальные) %,Target = 0 (Сгенерированные) %
eq_clean,,
Bands,0.14,3.15
Barbell,17.51,14.56
Bodyweight,10.55,19.31
Cardio,2.11,2.24
Dumbbell,13.85,15.22
Machine,54.25,30.39
Other,1.58,15.12



📊 Распределение признака: LEVEL_IDX


,Target = 1 (Реальные) %,Target = 0 (Сгенерированные) %
level_idx,,
2,93.95,51.57
0,5.06,37.06
1,0.91,6.80
3,0.07,4.57



📊 Распределение признака: GOAL_CLEAN


,Target = 1 (Реальные) %,Target = 0 (Сгенерированные) %
goal_clean,,
Athletics,1.55,12.66
Bodybuilding,90.15,30.05
Bodyweight Fitness,0.70,1.38
Fitness,0.00,7.32
Muscle & Sculpting,5.38,19.60
Powerbuilding,1.16,14.82
Powerlifting,1.05,14.17



📊 Распределение признака: IS_SAME_GROUP


,Target = 1 (Реальные) %,Target = 0 (Сгенерированные) %
is_same_group,,
0,10.58,67.72
1,89.42,32.28



📈 Статистика для BERT_SCORE:


,count,mean,std,min,25%,50%,75%,max
target,,,,,,,,
0,50000.0,0.00142,0.00029,0.00103,0.00124,0.00134,0.00153,0.00464
1,2844.0,0.00143,0.00027,0.00110,0.00124,0.00136,0.00154,0.00274


In [9]:
df_train['bert_score'].describe()

count    52844.000000
mean         0.001425
std          0.000285
min          0.001033
25%          0.001235
50%          0.001340
75%          0.001527
max          0.004641
Name: bert_score, dtype: float64

In [11]:
# Смотрим размер до чистки
print(f"Размер до удаления 'Other': {len(df_train)} строк")

# Выжигаем весь Other (и из нулей, и из единиц)
df_train = df_train[df_train['eq_clean'] != 'Other'].reset_index(drop=True)

# Смотрим размер после
print(f"Размер после зачистки: {len(df_train)} строк")

# Пересоздаем X и y для обучения (чтобы CatBoost получил чистые данные)
X_train, X_test, y_train, y_test = train_test_split(
    df_train.drop('target', axis=1), 
    df_train['target'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df_train['target']
)
print("✅ Мусор удален! Данные пересобраны.")

Размер до удаления 'Other': 52844 строк
Размер после зачистки: 45239 строк
✅ Мусор удален! Данные пересобраны.


In [ ]:
# Укрупняем категории инвентаря
# Объединяем Резинки, Кардио и Свой вес в одну супер-группу
inventory_map = {
    'Bands': 'Bodyweight',
    'Cardio': 'Bodyweight',
    'Bodyweight': 'Bodyweight',
    'Machine': 'Machine',
    'Dumbbell': 'Dumbbell',
    'Barbell': 'Barbell'
}

df_clean['eq_clean'] = df_clean['eq_clean'].map(inventory_map)

print("✅ Категории укрупнены. Теперь Кардио и Резинки = Bodyweight")
print(df_clean['eq_clean'].value_counts())

In [13]:
# Сохраняем финальный датафрейм, чтобы завтра не делать это заново
df_train.to_csv('rank_dataset_TRAIN_CATBOOST.csv', index=False)
print("💾 Файл 'rank_dataset_TRAIN_CATBOOST.csv' успешно сохранен! Можно переходить к обучению.")

💾 Файл 'rank_dataset_TRAIN_CATBOOST.csv' успешно сохранен! Можно переходить к обучению.
